# Hermes voice agent — Qwen3-ASR → Claude → gestalt world

Speak a command; an agent acts on a sandbox 3D world.

```
  mic  ──►  Qwen3-ASR (GPU)  ──►  text  ──►  Claude (claude-opus-4-8)
                                                     │  tool calls (MCP, stdio)
                                                     ▼
                                            gestalt-mcp server
                                            (sandbox world: add / move /
                                             recolor / describe objects)
                                                     │
                                                     ▼  /content/world.json
                                            matplotlib 3D view
```

**Why Colab:** Qwen3-ASR runs locally on the notebook's GPU. Set the runtime to **GPU** (Runtime → Change runtime type → GPU) before running.

The agent loop lives in `run_command(text)` — it doesn't care whether `text` was typed or transcribed, which is the seam a *Hermes* agent will later wrap.

## 1. Runtime & install

In [ ]:
!nvidia-smi -L  # confirm a GPU is attached
%pip install -q -U "qwen-asr" "anthropic[mcp]" "mcp>=1.20" soundfile

In [ ]:
# Install the gestalt package (provides the `gestalt.mcp_server` sandbox-world MCP server).
import os
if not os.path.exists('gestalt'):
    !git clone -q https://github.com/davechendatascience/gestalt.git
%pip install -q -e ./gestalt

## 2. Anthropic API key

Add `ANTHROPIC_API_KEY` in Colab's **🔑 Secrets** panel (left sidebar), or paste it when prompted.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key: ')
print('Key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))

## 3. Load Qwen3-ASR on the GPU

`Qwen/Qwen3-ASR-0.6B` is small and fast; switch `ASR_MODEL_ID` to `Qwen/Qwen3-ASR-1.7B` for higher accuracy. First run downloads the weights.

In [ ]:
import torch
from qwen_asr import Qwen3ASRModel

ASR_MODEL_ID = 'Qwen/Qwen3-ASR-0.6B'  # or 'Qwen/Qwen3-ASR-1.7B'
asr = Qwen3ASRModel.from_pretrained(
    ASR_MODEL_ID,
    dtype=torch.bfloat16,   # float16 also works (e.g. on T4)
    device_map='cuda:0',
    max_new_tokens=256,
)

def transcribe(audio):
    """audio: a file path, URL, or (np.ndarray, sample_rate) tuple."""
    results = asr.transcribe(audio=audio, language=None)  # language=None -> auto-detect
    return results[0].text.strip()

print('ASR ready:', ASR_MODEL_ID)

## 4. Capture a voice command

`record(seconds)` records from your browser mic; `upload()` takes an audio file instead. Both return a 16 kHz mono WAV path. (You can skip this section and just type commands in step 5.)

In [ ]:
import subprocess
from base64 import b64decode
from IPython.display import Javascript, display

_RECORD_JS = '''
async function recordAudio(ms) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(res => rec.onstop = res);
  rec.start();
  await new Promise(r => setTimeout(r, ms));
  rec.stop();
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  const buf = await new Blob(chunks).arrayBuffer();
  const bytes = new Uint8Array(buf);
  let binary = '';
  for (let i = 0; i < bytes.length; i++) binary += String.fromCharCode(bytes[i]);
  return btoa(binary);
}
'''

def _to_wav(src, out_path='/content/command.wav'):
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', src,
                    '-ar', '16000', '-ac', '1', out_path], check=True)
    return out_path

def record(seconds=5, out_path='/content/command.wav'):
    from google.colab import output
    display(Javascript(_RECORD_JS))
    print(f'Recording {seconds}s — speak now...')
    b64 = output.eval_js(f'recordAudio({int(seconds * 1000)})')
    with open('/content/command.webm', 'wb') as f:
        f.write(b64decode(b64))
    print('Transcribing...')
    return _to_wav('/content/command.webm', out_path)

def upload(out_path='/content/command.wav'):
    from google.colab import files
    up = files.upload()
    return _to_wav(next(iter(up)), out_path)

In [ ]:
# Try it: speak something like
#   "add a red cube called box at one two zero, then a blue sphere above it"
wav = record(6)
command = transcribe(wav)
print('Heard:', command)

## 5. Wire Claude to the gestalt world (MCP over stdio)

`run_command(text)` launches the `gestalt-mcp` server as a subprocess, exposes its tools to Claude, and runs one turn. The world is persisted to `/content/world.json`, so state survives across calls and we can visualize it. A short conversation history is kept for references like *"move it up"*.

In [ ]:
import os, sys
from anthropic import AsyncAnthropic
from anthropic.lib.tools.mcp import async_mcp_tool
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

os.environ['GESTALT_WORLD_FILE'] = '/content/world.json'
if os.path.exists('/content/world.json'):
    os.remove('/content/world.json')  # start empty

client = AsyncAnthropic()
MODEL = 'claude-opus-4-8'
SYSTEM = (
    'You are Hermes, an assistant that manipulates a 3D sandbox world through '
    'tools. Commands are transcribed from speech, so tolerate minor mis-hearings '
    "and pick the most plausible intent (e.g. 'q' might be 'cube'). Use the tools "
    'to carry out each command, then confirm what you did in one short sentence. '
    'If a command refers to something implicitly, call describe_scene to re-ground.'
)
history = []  # [{'role': 'user'|'assistant', 'content': str}, ...]

def _server_params():
    return StdioServerParameters(
        command=sys.executable, args=['-m', 'gestalt.mcp_server'], env=dict(os.environ)
    )

async def run_command(text, verbose=True):
    async with stdio_client(_server_params()) as (read, write):
        async with ClientSession(read, write) as mcp_client:
            await mcp_client.initialize()
            tools = (await mcp_client.list_tools()).tools
            messages = history + [{'role': 'user', 'content': text}]
            runner = client.beta.messages.tool_runner(
                model=MODEL,
                max_tokens=2048,
                thinking={'type': 'adaptive'},
                system=SYSTEM,
                messages=messages,
                tools=[async_mcp_tool(t, mcp_client) for t in tools],
            )
            final = None
            async for message in runner:
                final = message
                if verbose:
                    for b in message.content:
                        if b.type == 'tool_use':
                            print(f'  · {b.name}({b.input})')
            reply = ''.join(b.text for b in final.content if b.type == 'text').strip() if final else ''
    history.append({'role': 'user', 'content': text})
    history.append({'role': 'assistant', 'content': reply})
    return reply

In [ ]:
# Deterministic demo (no mic needed) — verifies the whole Claude -> MCP loop.
reply = await run_command('Add a red cube named box at (1, 2, 0) and a blue sphere named ball at (1, 2, 3). How far apart are they?')
print('Hermes:', reply)

In [ ]:
# Run the command you spoke in step 4:
print('Hermes:', await run_command(command))

## 6. Visualize the world

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def show_world(path='/content/world.json'):
    try:
        objs = json.load(open(path))['objects']
    except FileNotFoundError:
        print('No world yet — run a command first.'); return
    if not objs:
        print('The world is empty.'); return
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')
    for o in objs:
        c = o.get('color') or 'gray'
        if not mcolors.is_color_like(c):
            c = 'gray'
        ax.scatter(o['x'], o['y'], o['z'], s=220, c=c, edgecolors='k', depthshade=True)
        ax.text(o['x'], o['y'], o['z'], f"  {o['name']} ({o['kind']})")
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')
    ax.set_title(f'gestalt world — {len(objs)} object(s)')
    plt.show()

show_world()

## 7. Full voice loop

Run this cell, speak a command, and watch the world update. Re-run it for each new command.

In [ ]:
async def voice_turn(seconds=6):
    text = transcribe(record(seconds))
    print('Heard:', text)
    print('Hermes:', await run_command(text))
    show_world()

await voice_turn()

## Where Hermes plugs in

- **Integration seam:** `run_command(text)` is the agent turn. A Hermes agent wraps this — feeding it transcribed (or typed) commands and surfacing the reply — without changing anything below it.
- **Transport-agnostic tools:** `gestalt-mcp` is a standard MCP server. It's launched here over **stdio**, but the same server can be exposed over HTTP/SSE and shared by Hermes or any other MCP client.
- **Ground truth lives in the world**, not the chat: `/content/world.json` is the state; `describe_scene` lets the agent re-ground at any time.

No GPU? Run the same loop with typed input on your own machine: `pip install -e ".[agent]"` then `python examples/text_agent.py`.